# FAISS Document Indexing with Robust Checkpointing

This notebook implements document indexing using FAISS with robust checkpointing and error handling. It includes:
- Safe resumption from interruptions
- Automatic backups
- Progress tracking
- Performance monitoring

## 1. Setup Environment and Imports

In [1]:
# Set CUDA device and import core libraries
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
torch.cuda.set_device(0)

# Import required libraries
import json
from pathlib import Path
import time
import shutil
from datetime import datetime, timedelta
from tqdm.notebook import tqdm, trange
from typing import Optional, Union, List, Dict, Set
from pydantic import BaseModel, Field
from collections import defaultdict

# Import LangChain components
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS

3


## 2. Initialize Vector Store Components

In [2]:
# Force CPU usage for embeddings
device = "cpu"
print("Using CPU for embeddings (GPU disabled)")

# Initialize embedding model with CPU only
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": device}
)

print(f"Embedding model initialized on CPU")
print("Embedding dimensions:", len(embeddings.embed_query("test")))

Using CPU for embeddings (GPU disabled)


/tmp/ipykernel_3849053/1354607249.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(


Embedding model initialized on CPU
Embedding dimensions: 384


## 3. Load and Process JSON Records

In [3]:
# Load JSON records from file
with open('filtered_records_on_zeolite.json', 'r') as file:
    filtered_records = json.load(file)

print(f"Loaded {len(filtered_records)} records from JSON file")

Loaded 42678 records from JSON file


## 4. Document Processing

In [4]:
# Process records into documents
documents = []
metadata = []

for record in filtered_records:
    doi = record.get("doi", "Unknown DOI")
    
    # Add abstract as a separate document
    abstract_text = record.get("abstract", "").strip()
    if abstract_text:
        combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
        documents.append(combined_text)
        metadata.append({"doi": doi, "source": "abstract"})
    
    # Add each paragraph as a separate document
    for para in record.get("paragraphs", []):
        paragraph_text = para.get("text", "").strip()
        if paragraph_text:
            combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
            documents.append(combined_text)
            metadata.append({"doi": doi, "source": "paragraph"})

print(f"Total documents processed: {len(documents)}")

Total documents processed: 1474439


## 5. Configure FAISS Index Parameters

In [5]:
# Configuration parameters
TESTING_MODE = False  # Set to True for testing (2 batches)
FORCE_RESTART = False  # Set to True to ignore existing checkpoints
BATCH_SIZE = 50  # Number of documents per batch

# File paths
save_directory = Path("./faiss_index")
backup_directory = Path("./faiss_index_backups")
checkpoint_file = save_directory / "checkpoint.json"

# Ensure directories exist
save_directory.mkdir(exist_ok=True)
backup_directory.mkdir(exist_ok=True)

print("Configuration set:")
print(f"- Save directory: {save_directory}")
print(f"- Backup directory: {backup_directory}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Testing mode: {TESTING_MODE}")
print(f"- Force restart: {FORCE_RESTART}")

Configuration set:
- Save directory: faiss_index
- Backup directory: faiss_index_backups
- Batch size: 50
- Testing mode: False
- Force restart: False


## 6. Build Vector Database with Checkpointing

This section contains the main indexing loop with:
- Progress tracking
- Safe checkpointing
- Automatic backups
- Error handling

In [6]:
# Main indexing function
def create_vector_database(documents, metadata, embeddings, start_idx=0):
    """Create or resume FAISS index creation with safe checkpointing."""
    vector_db = None
    total = len(documents)
    
    # Try loading existing database
    if (save_directory / "index.faiss").exists() and start_idx > 0:
        try:
            print(f"Loading existing database from {save_directory}")
            vector_db = FAISS.load_local(
                str(save_directory),
                embeddings,
                allow_dangerous_deserialization=True  # We trust our own files
            )
            print(f"Loaded {vector_db.index.ntotal} vectors")
        except Exception as e:
            print(f"Could not load existing database: {e}")
            vector_db = None
    
    # Process documents in batches
    with tqdm(total=total-start_idx, desc="Processing documents") as pbar:
        for batch_start in range(start_idx, total, BATCH_SIZE):
            batch_end = min(batch_start + BATCH_SIZE, total)
            batch_docs = documents[batch_start:batch_end]
            batch_meta = metadata[batch_start:batch_end]
            
            try:
                # First batch: create new DB
                if vector_db is None:
                    print("\nCreating new vector database...")
                    vector_db = FAISS.from_texts(batch_docs, embeddings, metadatas=batch_meta)
                # Subsequent batches: add to existing
                else:
                    vector_db.add_texts(batch_docs, metadatas=batch_meta)
                
                # Backup and save after each batch
                if vector_db is not None:
                    # Backup existing files
                    if (save_directory / "index.faiss").exists():
                        backup_dir = backup_directory / f"backup_{batch_end}"
                        backup_dir.mkdir(exist_ok=True)
                        for f in ["index.faiss", "index.pkl"]:
                            if (save_directory / f).exists():
                                shutil.copy2(save_directory / f, backup_dir / f)
                    
                    # Save new state
                    vector_db.save_local(str(save_directory))
                    
                    # Update checkpoint
                    with open(checkpoint_file, 'w') as f:
                        json.dump({
                            'last_processed': batch_end,
                            'total_documents': total,
                            'timestamp': time.time()
                        }, f)
                    
                    print(f"\nSaved checkpoint: {batch_end}/{total} documents")
                
                # Update progress
                pbar.update(len(batch_docs))
                
            except KeyboardInterrupt:
                print("\nInterrupted! Saving progress...")
                if vector_db is not None:
                    vector_db.save_local(str(save_directory))
                raise
            
            except Exception as e:
                print(f"\nError in batch {batch_start}-{batch_end}: {e}")
                print("Attempting to continue...")
                continue
    
    return vector_db

In [ ]:
# Load checkpoint and start indexing
start_idx = 0
if checkpoint_file.exists() and not FORCE_RESTART:
    with open(checkpoint_file) as f:
        checkpoint = json.load(f)
        start_idx = checkpoint['last_processed']
        print(f"Resuming from document {start_idx}")

# Create/resume vector database
vector_db = create_vector_database(documents, metadata, embeddings, start_idx)
print(f"Final database has {vector_db.index.ntotal} vectors")

Resuming from document 577150
Loading existing database from faiss_index
Could not load existing database: Error in void faiss::read_xb_vector(VectorT&, IOReader*) [with VectorT = MaybeOwnedVector<unsigned char>] at /home/runner/miniconda3/conda-bld/faiss-pkg_1745590516582/work/faiss/impl/index_read.cpp:191: Error: 'ret == (size)' failed: read error in faiss_index/index.faiss: 23453651 != 26342400 (Success)


Processing documents:   0%|          | 0/897289 [00:00<?, ?it/s]


Creating new vector database...

Saved checkpoint: 577200/1474439 documents

Saved checkpoint: 577250/1474439 documents

Saved checkpoint: 577300/1474439 documents

Saved checkpoint: 577350/1474439 documents

Saved checkpoint: 577400/1474439 documents

Saved checkpoint: 577450/1474439 documents

Saved checkpoint: 577500/1474439 documents

Saved checkpoint: 577550/1474439 documents

Saved checkpoint: 577600/1474439 documents

Saved checkpoint: 577650/1474439 documents

Saved checkpoint: 577700/1474439 documents

Saved checkpoint: 577750/1474439 documents

Saved checkpoint: 577800/1474439 documents

Saved checkpoint: 577850/1474439 documents

Saved checkpoint: 577900/1474439 documents

Saved checkpoint: 577950/1474439 documents

Saved checkpoint: 578000/1474439 documents

Saved checkpoint: 578050/1474439 documents

Saved checkpoint: 578100/1474439 documents

Saved checkpoint: 578150/1474439 documents

Saved checkpoint: 578200/1474439 documents

Saved checkpoint: 578250/1474439 documents

## 7. Database Statistics and Recovery

In [ ]:
# Calculate database statistics
if vector_db is not None:
    embedding_dim = len(embeddings.embed_query("test"))
    num_vectors = vector_db.index.ntotal
    approx_size_per_vector = 4 * embedding_dim  # 4 bytes per float
    
    approx_faiss_size_mb = (num_vectors * approx_size_per_vector) / (1024 * 1024)
    approx_metadata_size_mb = len(json.dumps(metadata)) / (1024 * 1024)
    total_size_mb = approx_faiss_size_mb + approx_metadata_size_mb
    
    print("\n--- Vector Database Statistics ---")
    print(f"Number of vectors: {num_vectors}")
    print(f"Embedding dimensions: {embedding_dim}")
    print(f"Approximate FAISS index size: {approx_faiss_size_mb:.2f} MB")
    print(f"Approximate metadata size: {approx_metadata_size_mb:.2f} MB")
    print(f"Estimated total size: {total_size_mb:.2f} MB")
    
print("\n--- Checkpoint Recovery Information ---")
print("To recover from interruption:")
print("1. Run this notebook again - it will resume automatically")
print("2. Set FORCE_RESTART = True to start fresh")
print(f"3. Backups are in: {backup_directory}")
print("4. Periodic backups occur every 5% progress")